# Phase 1: GuideLLM Performance Benchmark

This notebook measures **inference performance** of your deployed LLM using [GuideLLM](https://github.com/neuralmagic/guidellm) through the EvalHub SDK. GuideLLM is a performance benchmarking platform designed to evaluate language model inference servers under realistic production conditions.

## What GuideLLM Measures

| Metric | Description |
|--------|-------------|
| **TTFT** | Time to First Token — latency before the first token arrives |
| **ITL** | Inter-Token Latency — time between consecutive tokens |
| **E2E Latency** | End-to-end request latency |
| **Throughput** | Tokens per second (output) |

## Available Profiles

| Profile | Use Case |
|---------|----------|
| `quick_perf_test` | Quick baseline (short duration) |
| `sweep` | Auto-discover optimal load — ramps rate until saturation |
| `throughput` | Maximum throughput discovery |
| `concurrent` | Fixed concurrency stress test |
| `constant` | Steady-state load at a fixed request rate |
| `poisson` | Realistic bursty traffic simulation |
| `comprehensive_perf_test` | Full characterization (all profiles) |

## Prerequisites

- **0_setup/0_model_deploy.ipynb** completed (model deployed)
- **0_setup/2_eval_hub_setup.ipynb** completed (EvalHub + MLflow running)
- `oc port-forward -n <namespace> svc/evalhub 8443:8443` (auto-handled by notebook)

## Step 1: Configuration

In [1]:
import os, subprocess
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "gemma4-e2b-deployment")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")

# GuideLLM needs the base /v1 endpoint (not /v1/completions)
GUIDELLM_URL = BASE_URL.replace("/v1/completions", "/v1").replace("/v1/chat/completions", "/v1")

import sys; sys.path.insert(0, '..')
from utils.port_forward import ensure_evalhub_port_forward
EVALHUB_URL = ensure_evalhub_port_forward(namespace=NAMESPACE)
_r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None

print(f"Namespace:         {NAMESPACE}")
print(f"Model Name:        {MODEL_NAME}")
print(f"GuideLLM Endpoint: {GUIDELLM_URL}")
print(f"EvalHub URL:       {EVALHUB_URL}")

EvalHub already reachable at localhost:8443
Namespace:         rhoai-models
Model Name:        qwen3-14b
GuideLLM Endpoint: https://qwen3-14b-kserve-workload-svc.rhoai-models.svc.cluster.local:8000/v1
EvalHub URL:       https://localhost:8443


## Step 2: Initialize EvalHub Client

In [2]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(url=GUIDELLM_URL, name=MODEL_NAME)

# Verify GuideLLM provider
providers = client.providers.list()
guidellm_found = any("GuideLLM" in getattr(p, "name", "") for p in providers)
print(f"GuideLLM provider registered: {guidellm_found}")

if guidellm_found:
    for p in providers:
        if "GuideLLM" in getattr(p, "name", ""):
            benchmarks = getattr(p, "benchmarks", [])
            print(f"Available benchmarks ({len(benchmarks)}):")
            for b in benchmarks:
                bid = getattr(b, "benchmark_id", getattr(b, "id", "?"))
                bname = getattr(b, "name", "?")
                print(f"  - {bid}: {bname}")

TLS verification disabled - skipping CA bundle detection


TLS verification disabled (insecure mode)


GuideLLM provider registered: True
Available benchmarks (7):
  - sweep: Auto-discover optimal load
  - throughput: Maximum throughput discovery
  - concurrent: Fixed concurrency stress test
  - constant: Steady-state load test
  - poisson: Realistic traffic simulation
  - quick_perf_test: Quick performance snapshot
  - comprehensive_perf_test: Full performance characterization


## Step 3: Quick Performance Baseline

Run a quick performance test to measure single-user latency (TTFT, ITL) with synthetic data.

In [3]:
from datetime import datetime

ts = datetime.now().strftime("%m%d-%H%M")

quick_request = JobSubmissionRequest(
    name=f"guidellm-quick-{MODEL_NAME}-{ts}",
    description=f"Quick performance baseline for {MODEL_NAME}",
    tags=["performance", "guidellm", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="constant",
            provider_id="guidellm",
            parameters={
                "rate": 2,
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-quick-{MODEL_NAME}-{ts}"),
)

quick_job = client.jobs.submit(quick_request)
print(f"Job submitted: {quick_job.id}")
print(f"State: {quick_job.state.value}")

Job submitted: 55c6e7ce-6f3d-48c8-9d4e-3af4e985c19e
State: pending


In [4]:
import time

job_id = quick_job.id
print(f"Waiting for job {job_id}...")

for i in range(60):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

Waiting for job 55c6e7ce-6f3d-48c8-9d4e-3af4e985c19e...


## Step 4: Rate Sweep Test

Automatically ramps up request rate to discover the saturation point — where latency begins to degrade. This identifies the maximum sustainable throughput.

In [5]:
sweep_request = JobSubmissionRequest(
    name=f"guidellm-sweep-{MODEL_NAME}-{ts}",
    description=f"Rate sweep to find saturation point for {MODEL_NAME}",
    tags=["performance", "guidellm", "sweep", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="sweep",
            provider_id="guidellm",
            parameters={
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-sweep-{MODEL_NAME}-{ts}"),
)

sweep_job = client.jobs.submit(sweep_request)
print(f"Sweep job submitted: {sweep_job.id}")
print(f"State: {sweep_job.state.value}")

Sweep job submitted: 7aac5110-5ffd-4c79-b4a4-81040456b79b
State: pending


In [6]:
job_id = sweep_job.id
print(f"Waiting for sweep job {job_id}...")

for i in range(120):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

Waiting for sweep job 7aac5110-5ffd-4c79-b4a4-81040456b79b...



Job completed!

Benchmark: sweep
MLflow Run: None
Metrics:
  mean_itl_ms: 0
  mean_ttft_ms: 0
  output_tokens_per_second: 24.50146498457749
  prompt_tokens_per_second: 47.84572299367599
  requests_per_second: 0.175


## Step 5: Constant Load Test

Sustain a fixed request rate to measure steady-state performance. Adjust the `rate` parameter based on your sweep results.

In [7]:
constant_request = JobSubmissionRequest(
    name=f"guidellm-constant-{MODEL_NAME}-{ts}",
    description=f"Constant load test at 2 req/s for {MODEL_NAME}",
    tags=["performance", "guidellm", "constant", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="constant",
            provider_id="guidellm",
            parameters={
                "rate": 2,
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
                "warmup": "5%",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-constant-{MODEL_NAME}-{ts}"),
)

constant_job = client.jobs.submit(constant_request)
print(f"Constant load job submitted: {constant_job.id}")
print(f"State: {constant_job.state.value}")

Constant load job submitted: f6304c15-eb0e-4217-98b2-7f92ff8c0c50
State: pending


In [8]:
job_id = constant_job.id
print(f"Waiting for constant load job {job_id}...")

for i in range(120):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

Waiting for constant load job f6304c15-eb0e-4217-98b2-7f92ff8c0c50...



Job completed!

Benchmark: constant
MLflow Run: None
Metrics:
  mean_itl_ms: 0
  mean_ttft_ms: 0
  output_tokens_per_second: 24.408517903879535
  prompt_tokens_per_second: 47.664218733463606
  requests_per_second: 0.1826058638782878


## Next Steps

- View all performance metrics in the **MLflow UI** (Experiments tab)
- Run **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb** for single Korean MCQ benchmark evaluation
- Run **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** for a unified evaluation (accuracy + performance) tracked in a single MLflow experiment
- Adjust `rate`, `max_seconds`, and `data` parameters based on your deployment scale